# Исследование детекции посторонней речи в двухканальной телефонной записи

Данный ноутбук содержит полный цикл исследования:
1. Демонстрация физической модели сигнала и генератора диалогов ()
2. Пошаговая визуализация искажений тракта (RIR, нелинейность динамика, задержка $\tau$, A-law, полоса 300–3400 Гц, packet loss)
3. Анализ базового правила (+6 дБ) и визуализация его ошибок (, , )
4. Подсчет метрик с допуском 50 мс на границах ()


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
sys.path.append(str(Path(".").resolve()))

import numpy as np
import matplotlib.pyplot as plt

from src.config import SAMPLE_RATE, FRAME_DURATION_SEC, COLLAR_SEC, LABEL_OWN, LABEL_LEAK, LABEL_THIRD
from src.synthesizer import TelephoneDialogueSynthesizer, SynthesisConfig
from src.baseline import EnergyBaselineDetector, predict_baseline_rule
from src.metrics import evaluate_channels, intervals_to_frame_matrix, precision_recall_f1

print("Environment initialized successfully. Sample rate:", SAMPLE_RATE, "Hz")


## 1. Физическая генерация диалога и телефонного тракта

Моделируем запись диалога длительностью 12 секунд со сценарием  и присутствием третьего говорящего ().

In [ ]:
synth = TelephoneDialogueSynthesizer(seed=42)
res = synth.synthesize_dialogue(duration_sec=12.0, scenario="double_talk", third_speaker_prob=1.0)

audio = res.audio
sr = res.sample_rate
gt = res.ground_truth_intervals

print(f"Synthesized stereo audio shape: {audio.shape}, duration: {res.duration_sec}s")
print("Metadata:", res.metadata)
print(f"GT intervals A ({len(gt['A'])}):", gt["A"][:3])
print(f"GT intervals B ({len(gt['B'])}):", gt["B"][:3])


## 2. Визуализация волновых форм, спектрограмм и Ground-Truth разметки

Отрисуем сигналы обоих каналов, их спектрограммы и временные интервалы меток (, , ).

In [ ]:
def plot_signal_and_labels(audio, sr, gt_intervals, title="Ground-Truth Timeline"):
    t = np.arange(audio.shape[1]) / sr
    fig, axes = plt.subplots(3, 2, figsize=(15, 8), sharex=True, gridspec_kw={"height_ratios": [1.2, 1.2, 1.0]})
    
    # Waveforms
    axes[0, 0].plot(t, audio[0], color="#1f77b4", lw=0.8)
    axes[0, 0].set_title("Track A (Left Microphone) - Waveform")
    axes[0, 0].set_ylabel("Amplitude")
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].plot(t, audio[1], color="#ff7f0e", lw=0.8)
    axes[0, 1].set_title("Track B (Right Microphone) - Waveform")
    axes[0, 1].grid(True, alpha=0.3)
    
    # Spectrograms
    axes[1, 0].specgram(audio[0], Fs=sr, NFFT=256, noverlap=128, cmap="viridis")
    axes[1, 0].set_title("Track A - Spectrogram (300–3400 Hz telephone band)")
    axes[1, 0].set_ylabel("Frequency (Hz)")
    
    axes[1, 1].specgram(audio[1], Fs=sr, NFFT=256, noverlap=128, cmap="viridis")
    axes[1, 1].set_title("Track B - Spectrogram (300–3400 Hz telephone band)")
    
    # Label Timelines
    color_map = {"own": "#2ca02c", "leak": "#d62728", "third": "#9467bd"}
    y_positions = {"own": 2, "leak": 1, "third": 0}
    
    for i, ch in enumerate(["A", "B"]):
        ax = axes[2, i]
        ax.set_ylim(-0.5, 2.5)
        ax.set_yticks([0, 1, 2])
        ax.set_yticklabels(["third", "leak", "own"])
        ax.set_xlabel("Time (seconds)")
        ax.grid(True, alpha=0.3)
        
        for seg in gt_intervals.get(ch, []):
            lbl = seg["label"]
            y_pos = y_positions.get(lbl, 0)
            ax.barh(y_pos, seg["end"] - seg["start"], left=seg["start"], height=0.6,
                    color=color_map.get(lbl, "gray"), alpha=0.85, edgecolor="black")
        ax.set_title(f"Ground-Truth Activity: Track {ch}")
    
    plt.tight_layout()
    plt.show()

plot_signal_and_labels(audio, sr, gt)


## 3. Оценка базового правила (+6 дБ) и визуализация его ошибок

Запустим базовое правило на сгенерированной записи и сравним его предсказания с Ground-Truth.

In [ ]:
baseline_detector = EnergyBaselineDetector()
baseline_preds = baseline_detector.predict_intervals(audio, sample_rate=sr)

# Подсчет метрик с допуском 50 мс на границах
metrics_results = evaluate_channels(
    ground_truth=gt,
    predictions=baseline_preds,
    num_frames=res.num_frames,
    frame_sec=FRAME_DURATION_SEC,
    collar_frames=5,  # 50 мс
)

print(f"Macro F1 (Baseline): {metrics_results['macro_f1']:.4f}")
print("Per-label F1 summary:")
for lbl, val in metrics_results["by_label"].items():
    print(f"  {lbl:5s}: F1 = {val['f1']:.4f}")

print("Detailed breakdown:")
for ch in ["A", "B"]:
    print(f"Channel {ch}:")
    for lbl in ["own", "leak", "third"]:
        m = metrics_results["by_channel"][ch][lbl]
        print(f"  {lbl:5s} -> Prec: {m['precision']:.3f}, Rec: {m['recall']:.3f}, F1: {m['f1']:.3f}")


## 4. Сравнение предсказаний базового правила с Ground-Truth

Сопоставим графически разметку базового правила и истинную разметку, чтобы наглядно увидеть места ошибок:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
color_map = {"own": "#2ca02c", "leak": "#d62728", "third": "#9467bd"}
y_positions = {"own": 2, "leak": 1, "third": 0}

# Channel A Ground Truth
axes[0].set_ylim(-0.5, 2.5)
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(["third", "leak", "own"])
axes[0].grid(True, alpha=0.3)
axes[0].set_title("Track A: Ground Truth")
for seg in gt.get("A", []):
    lbl = seg["label"]
    axes[0].barh(y_positions[lbl], seg["end"] - seg["start"], left=seg["start"], height=0.6,
                 color=color_map.get(lbl, "gray"), alpha=0.85, edgecolor="black")

# Channel A Baseline Prediction
axes[1].set_ylim(-0.5, 2.5)
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(["third", "leak", "own"])
axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel("Time (seconds)")
axes[1].set_title("Track A: Baseline Rule (+6 dB) Predictions")
for seg in baseline_preds.get("A", []):
    lbl = seg["label"]
    axes[1].barh(y_positions[lbl], seg["end"] - seg["start"], left=seg["start"], height=0.6,
                 color=color_map.get(lbl, "gray"), alpha=0.85, edgecolor="black")

plt.tight_layout()
plt.show()
